In [ ]:
# 学習
import torch
import torch.optim as optim
from yolov3.models.yolo import Model
import yaml
from tqdm import tqdm
from src.domain.loss import CustomLoss
from src.domain.dataloader import CustomDataset, custom_collate_fn
from torch.utils.data import DataLoader

# モデルのロード
config_path = 'yolov3/models/yolov5s.yaml'
model_path = 'models/pre_trained/yolov5s.pt'
model = Model(config_path)
model.load_state_dict(torch.load(model_path)['model'].state_dict())  # yolov5s.ptは、学習済みの重みファイル
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model.to(device)

hyp_path = "data/hyps/hyp.scratch-low.yaml"
with open(hyp_path, errors="ignore") as f:
    hyp = yaml.safe_load(f)

model.hyp = hyp

# カスタム損失関数
criterion = CustomLoss(model)

# オプティマイザ
optimizer = optim.Adam(model.parameters(), lr=0.001)
# scaler = torch.cuda.amp.GradScaler(enabled=True) # 高速化ライブラリ必要であれば利用したい

# データローダ
img_dir = './data/coco128/images/train2017'
annotation_dir = './data/coco128/labels/train2017'
train_dataset = CustomDataset(
    img_dir=img_dir,
    annotation_dir=annotation_dir,
)
train_loader = DataLoader(train_dataset, batch_size=4, shuffle=True, collate_fn=custom_collate_fn)

# tqdmの表示フォーマット
TQDM_BAR_FORMAT = '{l_bar}{bar:10}{r_bar}'

# トレーニングループ
num_epochs = 10

for epoch in range(num_epochs):
    model.train()
    running_loss = 0.0
    pbar = tqdm(train_loader, total=len(train_loader), bar_format=TQDM_BAR_FORMAT)
    for images, targets in pbar:
        images, targets = images.to(device), targets.to(device)
        optimizer.zero_grad()
        outputs = model(images)
        loss, loss_items = criterion(outputs, targets)
        # scaler.scale(loss).backward()
        loss.backward()
        optimizer.step()
        running_loss += loss.item()
        pbar.set_description(f'Epoch [{epoch + 1}/{num_epochs}], Loss: {running_loss / len(train_loader)}')
    print(f'Epoch [{epoch + 1}/{num_epochs}], Loss: {running_loss / len(train_loader)}')
    

# トレーニング済みモデルの保存
torch.save(model.state_dict(), 'models/fine_tuned/yolov5s_finetuned.pth')
print("model save!")


In [4]:
# 推論
import torch
import torch.optim as optim
from yolov3.models.yolo import Model
import yaml
from tqdm import tqdm
from src.domain.loss import CustomLoss
from src.domain.dataloader import CustomDataset, custom_collate_fn
from torch.utils.data import DataLoader

from torchvision import transforms
from PIL import Image

# モデルのロード
config_path = 'yolov3/models/yolov5s.yaml'
model_path = 'models/fine_tuned/yolov5s_finetuned.pth'
model = Model(config_path)
model.load_state_dict(torch.load(model_path))
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model.to(device)

# 推論モード
model.eval()

image_path = "data/images/bus.jpg"
image = Image.open(image_path).convert("RGB")
transform = transforms.Compose([
            transforms.Resize((576, 576)),
            transforms.ToTensor(),
        ])
image = transform(image)
image = image.to(device)
with torch.no_grad():  # 勾配計算を無効化
    print(image.dim())
    output = model(image)


                 from  n    params  module                                  arguments                     
  0                -1  1      3520  models.common.Conv                      [3, 32, 6, 2, 2]              
  1                -1  1     18560  models.common.Conv                      [32, 64, 3, 2]                
  2                -1  1     18816  models.common.C3                        [64, 64, 1]                   
  3                -1  1     73984  models.common.Conv                      [64, 128, 3, 2]               
  4                -1  2    115712  models.common.C3                        [128, 128, 2]                 
  5                -1  1    295424  models.common.Conv                      [128, 256, 3, 2]              
  6                -1  3    625152  models.common.C3                        [256, 256, 3]                 
  7                -1  1   1180672  models.common.Conv                      [256, 512, 3, 2]              
  8                -1  1   1182720  

YOLOv3s summary: 214 layers, 7235389 parameters, 7235389 gradients, 16.6 GFLOPs



3


TypeError: conv2d() received an invalid combination of arguments - got (list, Parameter, NoneType, tuple, tuple, tuple, int), but expected one of:
 * (Tensor input, Tensor weight, Tensor bias, tuple of ints stride, tuple of ints padding, tuple of ints dilation, int groups)
      didn't match because some of the arguments have invalid types: (!list!, !Parameter!, !NoneType!, !tuple!, !tuple!, !tuple!, int)
 * (Tensor input, Tensor weight, Tensor bias, tuple of ints stride, str padding, tuple of ints dilation, int groups)
      didn't match because some of the arguments have invalid types: (!list!, !Parameter!, !NoneType!, !tuple!, !tuple!, !tuple!, int)
